# Unit 11 exercises: neural reranking (learning to rank the candidate pool)

Run the recsys data generators before you begin. Every path the course has built -- popularity, BM25 lexical, item-item, matrix factorization, semantic, two-tower -- is a **retrieval** path: a cheap candidate generator that sees one slice of the signal. Unit 6 blended their calibrated scores with **fixed** weights; Unit 10 fused a sparse + dense pair. This unit keeps that two-stage shape -- cheap retrieval proposes a candidate **pool**, an expensive model re-scores only that pool -- but replaces the fixed ordering key with a **learned** one: a tiny PyTorch MLP (`NeuralRerankerPath`) that scores each pooled candidate from a 17-dim **feature vector**.

Three honest headlines frame the unit. First, the reranker only **re-orders the pool**, so retrieval sets a hard **recall ceiling** (0.772 of eligible readers have a relevant book in their pool-50; no reranker's hit@10 can exceed that). Second, the **naive** training recipe **leaks and tanks**: label the reranker with a reader's train positives while the feature-paths were fit on that *same* train, and the paths have memorized those items -- the reranker learns inflated in-train scores that validation never shows, dropping to ~0.28 hit@10, *below* plain score-order. The fix is a **time-ordered holdout inside train** (latest ~25% of a reader's positives = labels; earlier 75% = the profile every training-time feature is built from). Third, the lift is **counterintuitive**: it comes from the **content** features (genre/author affinity, log-popularity), not from cleverly combining the path scores -- content-only reranking is the strongest variant and `linear ~= MLP`. The reranker edges the two-tower / ties the hybrid on hit@10 but **loses catalog coverage**, and you will read that honestly.

Everything is measured on the **validation** split (`val`); each reader's **seen** set is their train-positive history. The sealed `test` holdout stays sealed -- no cell here may touch it. Training is seeded and CPU-deterministic (`seed=0`). Each answer is left for you to complete; no solutions live here.

A speed note, because fitting recurs. The six retrieval paths are fit **once** in the setup cell and refit **once** on the 75% profile; the leakage-safe feature matrix is assembled **once** (the ~15 s step); and every model you train below fits on that cached matrix in well under a second. Reuse `serving`, `profile_paths`, `path_factory`, and `reranker` across the exercises -- do not refit.

In [ ]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    HybridRetrievalPath,
    ItemItemRetrievalPath,
    LexicalRetrievalPath,
    MatrixFactorizationPath,
    NeuralRerankerPath,
    PathRegistry,
    PopularityRetrievalPath,
    RerankerModel,
    SemanticEmbeddingRetrievalPath,
    TwoTowerRetrievalPath,
    run_blended_scoreboard,
)
from bookrec.rerank import family_mask, feature_names, split_profile_labels
from bookrec.scoreboard import _seen_and_relevant

SEED = 0
K = 10                 # every scoreboard reports at the top-10 cutoff
POOL = 50              # pinned: the pool recall ceiling is ~0.772 @ pool 50 vs ~0.716 @ pool 30
PATH_NAMES = ("popularity", "lexical", "item-item", "mf", "semantic", "two-tower")

# --- the real interaction log + content (the same substrate as Units 1-10) -----------------
data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")      # {item_id: Book} (2000 books)
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")   # {item_id: "space joined tokens"}
glove = bookrec.load_glove_subset()                              # the gensim-free GloVe subset (Unit 7)

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])     # readers the personalized paths cannot serve

interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")       # plain row dicts the paths duck-type

# Each reader's train-positive history is their `seen` (exclusion) set. We grade on `val`.
seen_by = defaultdict(set)
for row in rows:
    if row["label"] == 1 and row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])


def fit_six(train_rows):
    """Fit the six retrieval paths whose calibrated scores feed the reranker (seed 0)."""
    return {
        "popularity": PopularityRetrievalPath().fit(train_rows, catalog=catalog_ids),
        "lexical": LexicalRetrievalPath(keywords).fit(train_rows, catalog=catalog_ids),
        "item-item": ItemItemRetrievalPath().fit(train_rows, catalog=catalog_ids),
        "mf": MatrixFactorizationPath(seed=SEED).fit(train_rows, catalog=catalog_ids),
        "semantic": SemanticEmbeddingRetrievalPath(keywords, glove).fit(train_rows, catalog=catalog_ids),
        "two-tower": TwoTowerRetrievalPath(seed=SEED).fit(train_rows, catalog=catalog_ids),
    }


# The SERVING paths: fit on the FULL train once. Every exercise reuses them (~60 s total; do once).
serving = fit_six(rows)

# The leakage-safe TRAINING profile: hold out each reader's latest ~25% of train positives as the
# reranker's LABELS; the earlier 75% is the PROFILE the feature-paths (and every content/popularity
# statistic) are built from. Refit the six paths on that 75% profile ONCE (shared below).
profile, labels = split_profile_labels(rows, 0.25)
profile_rows = [
    r
    for r in rows
    if not (
        r["split"] == "train"
        and int(r["label"]) == 1
        and int(r["item_id"]) in labels.get(int(r["reader_id"]), set())
    )
]
profile_paths = fit_six(profile_rows)


def path_factory(train_rows):
    """The profile refit is done once up front; the reranker reuses it (passed via train_paths)."""
    return profile_paths


# The shared reranker over the serving paths (pool=50). You assemble its training matrix in
# Exercise 3 and install a learned model in Exercise 4; later exercises reuse this one object.
reranker = NeuralRerankerPath(serving, path_factory, catalog, pool=POOL, seed=SEED)


def build_pool(paths, reader, seen, top=POOL):
    """A reader's candidate POOL: {item_id: {path_name: calibrated_score}} over the paths' top-`top`.

    `calibrate` maps each path's scores to [0, 1] so the six very different score scales become
    comparable; the pool is the UNION of the six calibrated top-`top` lists keyed by item id.
    """
    per_item = defaultdict(dict)
    context = {"seen": seen}
    for name in PATH_NAMES:
        path = paths[name]
        for cand in path.calibrate(path.retrieve(reader, context, top)):
            per_item[cand.item_id][name] = cand.score
    return per_item


def board(path):
    """hit@10 + catalog coverage on `val` via the single-path blended scoreboard (reproduces hit@k)."""
    result = run_blended_scoreboard(
        {path.name: path}, interactions_path,
        catalog_ids=catalog_ids, k=K, pool=K, cold_readers=cold_readers,
    )
    return result.hit_rate_at_k, result.catalog_coverage

## Exercise 1 - Build the candidate pool and read the 17-dim feature vector

Retrieve-then-rank has two stages. The **retrieval** stage is cheap and proposes a candidate **pool**: the union of the six paths' top-`POOL` calibrated lists (`build_pool`, in the setup cell). The **rank** stage is expensive and precise -- it scores *only* that pool, so the pool is all the reranker ever sees.

For each pooled candidate the reranker builds a **17-dim feature vector**: the **6** calibrated per-path scores (an absent path contributes 0.0), **6** presence flags (was this item in that path's pool?), **1** `n_paths` meta feature, and **4** content signals (`genre_frac`, `genre_cos` of the candidate's genres against the reader's history, `author_frac`, `log_pop`). `feature_names(PATH_NAMES)` returns that ordered layout; `NeuralRerankerPath` assembles the full matrix for you (do not hand-build it).

Build one reader's pool, confirm its size is between `POOL` and `6 * POOL`, and read the feature layout -- confirm it is 17 wide and splits into 6 score + 6 presence + 1 meta + 4 content columns. In two or three sentences, explain the two-stage split and why the reranker sees only the pool.

In [ ]:
one_reader = sorted(seen_by)[0]          # the first reader with a train history

pool = None           # TODO: build_pool(serving, one_reader, seen_by[one_reader])
pool_size = None      # TODO: len(pool)   -- the union of the six paths' pools
pool_in_range = None  # TODO: bool - POOL <= pool_size <= 6 * POOL

# Read the feature layout (do NOT hand-build the vector -- the reranker assembles the matrix).
spec = None               # TODO: feature_names(PATH_NAMES)   -- the 17 ordered feature names
n_features = None         # TODO: len(spec)   -- 17
n_score = None            # TODO: sum(1 for s in spec if s.startswith("score:"))   -- 6 per-path scores
n_presence = None         # TODO: sum(1 for s in spec if s.startswith("in:"))      -- 6 presence flags
n_content = None          # TODO: sum(1 for s in spec if s in ("genre_frac", "genre_cos", "author_frac", "log_pop"))  -- 4
layout_adds_up = None     # TODO: bool - n_score + n_presence + 1 + n_content == n_features == 17

pool_explanation = "TODO"  # 2-3 sentences: cheap retrieval proposes the POOL (the union of the six
                           # paths' calibrated top-POOL lists); the reranker only RE-ORDERS that pool,
                           # scoring each candidate from the 17-dim vector -- so the pool is all the
                           # reranker can ever recommend from

## Exercise 2 - The pool recall ceiling: retrieval caps the reranker

Because the reranker only re-orders the pool, it can recommend a relevant book **only if retrieval already put that book in the pool**. The measure is **any-relevant-in-pool recall**: the fraction of eligible `val` readers whose pool holds at least one of their held-out relevant books. That number is a hard **ceiling** on hit@10 -- no reranking, however clever, can exceed it.

Compute the ceiling at pool 50 and pool 30 and confirm the bigger pool lifts it (~0.772 vs ~0.716). This is exactly why `POOL = 50` is pinned: a larger pool buys ~0.05 of ceiling for a tiny extra cost. In two or three sentences, explain what the ceiling is and why it bounds every reranker.

In [ ]:
def pool_recall(paths, top):
    """Any-relevant-in-pool recall over the eligible `val` readers -- the reranker's hit@10 ceiling."""
    known = set(catalog_ids)
    seen_lookup, target_lookup = _seen_and_relevant(interactions_path, known, "val")
    cold = set(cold_readers)
    hits = []
    for reader in sorted(target_lookup):
        seen = seen_lookup[reader]
        relevant = target_lookup[reader] - seen
        if reader in cold or not relevant:
            continue
        per_item = build_pool(paths, reader, seen, top)
        hits.append(1.0 if any(i in relevant for i in per_item) else 0.0)
    return float(np.mean(hits)) if hits else 0.0


ceiling_50 = None   # TODO: pool_recall(serving, 50)   -- ~0.772
ceiling_30 = None   # TODO: pool_recall(serving, 30)   -- ~0.716
bigger_pool_lifts_ceiling = None  # TODO: bool - ceiling_50 > ceiling_30

ceiling_explanation = "TODO"  # 2-3 sentences: the pool recall ceiling is the fraction of readers
                              # whose pool contains a relevant book; the reranker only re-orders the
                              # pool, so its hit@10 can never exceed this -- retrieval caps reranking

## Exercise 3 - The leakage-safe time-ordered holdout

Where do the reranker's training labels come from? Not the retrieval pool directly -- retrieval *excludes* a reader's `seen` (their train positives), so the pool has no positives to learn from. The fix is a **time-ordered holdout inside train** (`split_profile_labels`): order each reader's train positives by time, hold out the latest ~25% as the reranker's **labels**, and keep the earlier 75% as the **profile**. Every training-time input -- the feature-paths *and* the genre/author/popularity statistics -- is built from that 75% profile only, so a path can never have trained on a label item. (That is the leak you will reproduce in Exercise 5.)

One honest subtlety: the leakage guarantee is **not** that the profile and label *sets* are disjoint. A reader who read the same book at an early *and* a late time has it in both the 75% profile and the 25% labels, so the sets can overlap (measure how often). The real invariant is that the training **pool excludes the profile** (`seen`), so a profile book is never a training candidate and the positive rows are exactly the held-out labels the profile never contained (`labels - profile`). Measure the repeat-read overlap, confirm a splittable reader has at least one genuinely-held-out label, assemble the leakage-safe `(X, y)` once (`reranker.assemble_training_matrix(rows, train_paths=profile_paths)`), and confirm `X` is 17 wide and `y` carries both classes. In three or four sentences, explain why training-time features come from the 75% profile and why the pool-excludes-profile step (not set disjointness) is what prevents the leak.

In [ ]:
prof, lab = None, None   # TODO: split_profile_labels(rows, 0.25)   -- {reader: profile set}, {reader: label set}

# A reader with >= 2 train positives is splittable (profile + >=1 held-out label). The <2-positive
# readers keep all positives as profile and supply no label -- they never train the reranker.
label_readers = None     # TODO: sorted(r for r in lab if lab[r] and prof.get(r))
some_reader = None       # TODO: label_readers[0]
has_profile_and_label = None  # TODO: bool - len(prof[some_reader]) >= 1 and len(lab[some_reader]) >= 1

# The raw profile and label SETS are NOT always disjoint -- a re-read book lands in both. Measure it.
overlapping = None            # TODO: [r for r in label_readers if (prof[r] & lab[r])]
overlap_fraction = None       # TODO: len(overlapping) / len(label_readers)       -- ~0.335
label_sets_always_disjoint = None  # TODO: bool - len(overlapping) == 0           -- FALSE here
# The REAL leakage invariant: positives are labels the profile never held (labels - profile); the
# training pool excludes the profile (seen), so those held-out labels are exactly what trains it.
new_labels = None             # TODO: lab[some_reader] - prof[some_reader]
has_genuinely_held_out_label = None  # TODO: bool - len(new_labels) >= 1

# Assemble the leakage-safe matrix ONCE (the ~15 s step). It also fixes the reranker's serving
# statistics, so set_model works afterward. Every exercise below reuses this X, y.
X, y = None, None        # TODO: reranker.assemble_training_matrix(rows, train_paths=profile_paths)
width_is_17 = None       # TODO: bool - X.shape[1] == 17
has_positives = None     # TODO: bool - bool((y == 1.0).any())
has_negatives = None     # TODO: bool - bool((y == 0.0).any())

holdout_explanation = "TODO"  # 3-4 sentences: retrieval excludes seen, so labels come from a
                              # time-ordered holdout inside train; the earlier 75% builds every
                              # training-time feature so no path trained on a label item; leakage-safety comes from the
                              # pool EXCLUDING the profile (not set disjointness); the held-out 25% supplies labels

## Exercise 4 - Train the reranker, register it, and read the val scoreboard

Now train the learned ordering key. Fit a tiny MLP on the cached matrix over **all** feature families (`family_mask(("scores", "presence", "meta", "content"))`), a `BCEWithLogits` pointwise-logistic ranker with Adam -- under a second. Install it with `reranker.set_model(...)` and register the path (it owns the name `reranker-v1`).

Read the `val` scoreboard on **both** hit@10 and catalog coverage, against the best single path (two-tower ~0.340), the U6 4-way blend (~0.306 / coverage ~0.333), the U10 hybrid (~0.362), and the fixed 6-way equal-weight **score-order** over the *same* pool (~0.348). The gate: the reranker must **not tank** (`>= two_tower - 0.01`) and must **beat the score-order over its own pool**. The honest cost: its coverage **collapses** well below the U6 blend. In three or four sentences, state where the reranker lands on hit@10 and read the coverage as a loss.

In [ ]:
# Train the clean-recipe reranker on the cached leakage-safe matrix and install it as the ordering key.
all_mask = None   # TODO: family_mask(("scores", "presence", "meta", "content"))
all_model = None  # TODO: RerankerModel(seed=SEED).fit(X, y, feature_mask=all_mask)
# TODO: reranker.set_model(all_model)   -- install the learned ordering key on the shared reranker

registry = PathRegistry()
# TODO: registry.register(reranker)
registered_name = None   # TODO: reranker.artifact_name()   -- "reranker-v1"

# The val scoreboard: hit@10 AND catalog coverage for every comparator.
reranker_hit, reranker_cov = None, None    # TODO: board(reranker)              -- ~0.36 / ~0.12-0.19
two_tower_hit, two_tower_cov = None, None  # TODO: board(serving["two-tower"])  -- ~0.340 / ~0.177

hybrid = HybridRetrievalPath(serving["lexical"], serving["two-tower"], weight=0.7, pool=50).fit(rows)
hybrid_hit, hybrid_cov = None, None        # TODO: board(hybrid)                -- ~0.362 / ~0.192

u6 = run_blended_scoreboard(
    {n: serving[n] for n in ("popularity", "lexical", "item-item", "mf")},
    interactions_path, catalog_ids=catalog_ids, k=K, pool=30,
    weights={"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0},
    cold_readers=cold_readers,
)
u6_hit, u6_cov = None, None                # TODO: u6.hit_rate_at_k, u6.catalog_coverage  -- ~0.306 / ~0.333

six_way = run_blended_scoreboard(
    {n: serving[n] for n in PATH_NAMES},
    interactions_path, catalog_ids=catalog_ids, k=K, pool=POOL, cold_readers=cold_readers,
)
six_way_hit = None    # TODO: six_way.hit_rate_at_k   -- ~0.348 (fixed equal-weight order over the pool)

not_tanked = None         # TODO: bool - reranker_hit >= two_tower_hit - 0.01   (gate a)
beats_score_order = None  # TODO: bool - reranker_hit >= six_way_hit            (gate b)
coverage_is_a_loss = None # TODO: bool - reranker_cov < u6_cov                  (honest: coverage collapses)

scoreboard_reading = "TODO"  # 3-4 sentences: the reranker edges the two-tower and ties the hybrid on
                             # hit@10 (and beats the fixed score-order over its own pool), but its
                             # catalog coverage COLLAPSES (~0.12-0.19 vs the U6 blend's ~0.333) -- an
                             # honest accuracy win at a real coverage cost

## Exercise 5 - The leakage trap: the naive recipe tanks

Here is the headline mistake. The **naive** recipe labels the reranker with a reader's train positives while the feature-paths were fit on that *same* full train -- so the paths have *memorized* the label items and score them inflatedly. The reranker dutifully learns "high score => positive", but that pattern is a training-set artifact: `val` candidates were never in any path's training set, so their scores are not inflated, and the learned rule misfires. The measured result is a **tank** to ~0.288 hit@10 -- below the clean recipe *and* below the plain 6-way score-order.

Reproduce it with `NeuralRerankerPath(..., leaky=True).fit(rows)` (this reranker does its own leaky fit; it does not touch your cached `X`). Board it on `val` and confirm it scores below both the clean reranker (`reranker_hit`, Exercise 4) and the score-order (`six_way_hit`, Exercise 4). In three or four sentences, explain *why* the naive recipe leaks and how the time-ordered holdout fixes it.

In [ ]:
leaky = NeuralRerankerPath(serving, path_factory, catalog, pool=POOL, seed=SEED, leaky=True)
# TODO: leaky.fit(rows)   -- trains on the full-fit serving paths + full-train counts (the leak)

leaky_hit, leaky_cov = None, None   # TODO: board(leaky)   -- ~0.288 (tanked)
# reranker_hit (clean recipe) and six_way_hit (fixed score-order) both come from Exercise 4.
leaky_below_clean = None        # TODO: bool - leaky_hit < reranker_hit
leaky_below_score_order = None  # TODO: bool - leaky_hit < six_way_hit

leak_explanation = "TODO"  # 3-4 sentences: the feature-paths memorized the label items, so in-train
                           # scores are inflated in a way val candidates never are (distribution
                           # shift); the learned "high score => positive" rule misfires on val and
                           # tanks; the time-ordered holdout builds training features from the 75%
                           # profile only, so no path ever trained on a label item

## Exercise 6 - Feature ablation: content carries the lift

Which features actually earn the reranker's accuracy? The counterintuitive answer: the **content** features, not the path scores. Retrain on two feature subsets (cheap -- the matrix is cached): **content-only** (`family_mask(("content",))`) and **scores-only** (`family_mask(("scores", "presence", "meta"))`). Install each on the shared `reranker` (`reranker.set_model(model)`) and board it. Content-only is the strongest variant (~0.384); dropping the path scores does **not** hurt, while scores-only falls back toward the plain score-order.

The reason: the generator's taste is author-following and genre-derived, a cross-reader signal the per-list min-max-calibrated path *scores* cannot carry. Confirm `content_hit >= scores_hit`. In three or four sentences, explain what the content features see that the path scores cannot, and what this says about "a learned ranker beats a blend".

In [ ]:
content_model = None   # TODO: RerankerModel(seed=SEED).fit(X, y, feature_mask=family_mask(("content",)))
scores_model = None    # TODO: RerankerModel(seed=SEED).fit(X, y, feature_mask=family_mask(("scores", "presence", "meta")))

# Install each model on the shared reranker, then board val. (set_model swaps the ordering key in
# place; re-install all_model from Exercise 4 if you want the full-feature reranker back afterward.)
content_hit, content_cov = None, None   # TODO: reranker.set_model(content_model); board(reranker)  -- ~0.384
scores_hit, scores_cov = None, None     # TODO: reranker.set_model(scores_model); board(reranker)    -- ~ score-order

content_at_least_scores = None   # TODO: bool - content_hit >= scores_hit

ablation_explanation = "TODO"  # 3-4 sentences: content-only is as strong or stronger than scores-only;
                               # the content features (genre/author affinity, log-popularity) express a
                               # cross-reader taste signal the per-list calibrated path scores cannot;
                               # the learned ranker wins by SEEING new features, not by combining scores

## Exercise 7 - Challenge: linear vs MLP (the non-linearity adds nothing)

If the lift is content *affinity* rather than a non-linear interaction of the path scores, a **linear** model should do just as well as the MLP. Train a single-`Linear`-layer reranker (`RerankerModel(seed=SEED, linear=True)`) over all feature families -- this is logistic regression, the Unit 6 blend's fixed weights now *learned* -- install it, and board it.

Compare its hit@10 to the MLP's (`reranker_hit` from Exercise 4) and confirm they are within ~0.1: `linear ~= MLP` on this data. Record this; it is never gated. In two or three sentences, explain what `linear ~= MLP` tells you about where the reranker's accuracy comes from.

In [ ]:
linear_model = None   # TODO: RerankerModel(seed=SEED, linear=True).fit(X, y, feature_mask=all_mask)

# Install and board (restore all_model from Exercise 4 afterward if you want the MLP back).
linear_hit, linear_cov = None, None   # TODO: reranker.set_model(linear_model); board(reranker)
# reranker_hit (the all-feature MLP) comes from Exercise 4.
linear_approx_mlp = None   # TODO: bool - abs(linear_hit - reranker_hit) < 0.1

linear_explanation = "TODO"  # 2-3 sentences: the single linear layer matches the MLP, so the
                             # non-linear hidden layer adds nothing here; the lift is a (near-)linear
                             # read of the content-affinity features, not a non-linear combination of
                             # correlated path scores

## Exercise 8 - Challenge: the pool-size knob and the recall ceiling

The pool size is a knob with a clear trade. A bigger pool raises the recall **ceiling** the reranker can reach (Exercise 2), at the cost of more candidates to assemble features for and score per reader. Explore it: compute the ceiling across a few pool sizes and confirm it rises with the pool, then confirm the pinned reranker's hit@10 (`reranker_hit`, Exercise 4) sits **under** its pool-50 ceiling -- the cap you cannot cross by reranking alone.

Reuse `pool_recall` from Exercise 2. In three or four sentences, describe the trade the pool-size knob makes and why the reranker's hit@10 can never beat its pool's ceiling -- and what that implies about where to spend effort if you want a *higher* hit@10.

In [ ]:
pool_sizes = [20, 30, 50]
ceiling_by_pool = None   # TODO: {p: pool_recall(serving, p) for p in pool_sizes}   -- rises with p
ceiling_rises_with_pool = None  # TODO: bool - ceiling_by_pool[20] <= ceiling_by_pool[50]

# reranker_hit (the pinned pool-50 reranker) comes from Exercise 4; it cannot beat its pool's ceiling.
reranker_under_ceiling = None   # TODO: bool - reranker_hit <= ceiling_by_pool[50]

pool_knob_explanation = "TODO"  # 3-4 sentences: a bigger pool lifts the recall ceiling at more
                                # feature/scoring cost; the reranker only re-orders the pool so its
                                # hit@10 is capped by the pool's ceiling; to raise hit@10 past the cap
                                # you must improve RETRIEVAL (a bigger/better pool), not the ranker